# Guardrails AI: validation patterns

A hands-on walkthrough of validating LLM text, handling failures, extracting structured data, and integrating Guardrails with LangChain.

Run the notebook from top to bottom. Later examples reuse the model configuration, constants, and validators defined in earlier cells.

## Contents

1. [Setup and configuration](#1-setup-and-configuration)
2. [PII detection](#2-pii-detection)
3. [Competitor checks](#3-competitor-check)
4. [Toxic language](#4-toxic-language)
5. [Topic restriction](#5-restrict-to-topic)
6. [Failure actions](#6-onfailactions)
7. [Structured extraction](#7-io-structure-validation-using-guardrailsai)
8. [LangChain integration](#8-langchain-x-guardrails-ai)
9. [Streaming validation](#9-guardrails-validation-during-streaming)

In [1]:
print("Hello, This is Guardian AI, your personal assistant. How can I help you today?")

Hello, This is Guardian AI, your personal assistant. How can I help you today?


## 1. Setup and configuration

Use the project virtual environment, set the Groq API key when prompted, and configure the model and assistant instructions before running the examples. The setup cell disables Guardrails tracing and filters known library noise; validation and model errors remain visible.

In [2]:
import getpass
import logging
import os
import warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["OTEL_SDK_DISABLED"] = "true"

warnings.filterwarnings(
    "ignore",
    message=r"Could not obtain an event loop\. Falling back to synchronous validation\.",
    category=UserWarning,
    module=r"guardrails\.validator_service",
)
warnings.filterwarnings(
    "ignore",
    message=r"IProgress not found\.",
    category=Warning,
    module=r"tqdm\.auto",
)
warnings.filterwarnings(
    "ignore",
    message=r"`torch\.jit\.script` is not supported in Python 3\.14\+.*",
    category=FutureWarning,
    module=r"torch\.jit\._script",
)

class _UnsupportedPresidioLanguageFilter(logging.Filter):
    def filter(self, record):
        return not (
            record.name == "presidio-analyzer"
            and record.getMessage().startswith(
                "Recognizer not added to registry because language is not supported by registry"
            )
        )

logging.getLogger("presidio-analyzer").addFilter(_UnsupportedPresidioLanguageFilter())

from guardrails.settings import settings

settings.disable_tracing = True

In [3]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)
print("Working dir:", os.getcwd())

Working dir: h:\2026\Learning\KrishNaik\AI_security_Evals_Gaurdrails\Guardrails_AI


In [4]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

In [5]:
BOT_MODEL = "groq/openai/gpt-oss-120b"  # litellm needs the groq/ provider prefix
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", BOT_MODEL)

Config ready: groq/openai/gpt-oss-120b


## 2. PII Detection 

In [6]:
from guardrails_ai.detect_pii import DetectPII
from guardrails import Guard, OnFailAction

In [7]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail=OnFailAction.FIX,
        use_local=True,     # light: Presidio + spaCy small, no torch
    )
)

In [8]:
raw = "We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out = pii_guard.validate(raw)
print(out)

ValidationOutcome[TypeVar](
    call_id='2385876028592',
    raw_llm_output="We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
    validation_summaries=[
        ValidationSummary(
            validator_name='DetectPII',
            validator_status='fail',
            property_path='$',
            failure_reason="The following text in your response contains PII:\nWe'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
            error_spans=[
                ErrorSpan(
                    start=18,
                    end=37,
                    reason='PII detected in 4111 1111 1111 1111'
                ),
                ErrorSpan(
                    start=63,
                    end=75,
                    reason='PII detected in jane@example'
                )
            ]
        )
    ],
    validated_output="We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.",
    reask=None,
    validation_

In [9]:
print("IN :", raw)
print("OUT:", out.validated_output)

IN : We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.
OUT: We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.


## 3. Competitor Check

The next examples compare a response with no competitor mentions to one that names configured competitors.

In [10]:
from guardrails_ai.competitor_check import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

### 3.1 Passing example: no competitor mentions

In [11]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out! We're happy to help with your refund."


### 3.2 Failing example: competitor mentions

In [12]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)
out = comp_guard.validate(BAD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than us, and [COMPETITOR] is easier to set up.But we're happy to help with your refund."


## 4. Toxic Language

In [13]:
from guardrails_ai.toxic_language import ToxicLanguage

In [14]:
tox_guard = Guard().use(
    ToxicLanguage(
        threshold=0.5,
        validation_method="sentence",
        on_fail=OnFailAction.FIX,
        use_local=True, 
    )
)

In [15]:
rude = "You are completely useless and your stupid fuck app stole my money."
out = tox_guard.validate(rude)
print("validation_passed:", out.validation_passed)
print("OUT:", repr(out.validated_output))

validation_passed: True
OUT: ''


In [16]:
out

ValidationOutcome[TypeVar](call_id='2386006928960', raw_llm_output='You are completely useless and your stupid fuck app stole my money.', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and your stupid fuck app stole my money.', error_spans=[ErrorSpan(start=0, end=67, reason='Toxic language detected: toxicity, obscene, insult')])], validated_output='', reask=None, validation_passed=True, error=None)

## 5. Restrict to Topic

In [17]:
import json, litellm
from guardrails_ai.restricttotopic import RestrictToTopic
from guardrails.errors import ValidationError

In [18]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    payload = json.loads(resp.choices[0].message.content)
    if not isinstance(payload, dict):
        raise ValueError("Topic classifier response must be a JSON object.")

    topics_present = payload.get("topics_present")
    if not isinstance(topics_present, list) or not all(
        isinstance(topic, str) for topic in topics_present
    ):
        raise ValueError("Topic classifier response must include a list of topic names.")

    return topics_present

In [19]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)



Device set to use cpu


In [20]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [21]:
out1 = None
try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out1.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print(str(e)[:300])


Raised ValidationError - off-topic refused.
Validation failed for field with errors: Invalid topics found: ['politics']


## 6. OnFailActions

We are performing Multiple OnFailActions on CompetitorCheck validator

from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from guardrails.errors import ValidationError

In [22]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from guardrails.errors import ValidationError

In [23]:
from guardrails_ai.competitor_check import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [24]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )

print("Input we'll validate five ways:\n", BAD_TEXT)

Input we'll validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.


### 6.1 FIX

In [25]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2386584551824',
    raw_llm_output="Thanks for reaching out! We're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! We're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We're happy to help with your refund.


In [26]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2386567821648',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees th

### 6.2 FILTER

In [27]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2386578020912',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

### 6.3 REFRAIN

In [28]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2386578017744',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

### 6.4 REFRAIN vs FILTER

In [29]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer": "Refunds settle within 24 hours.",'
    ' "comparison": "Honestly Stripe is cheaper than us.",'   # <- this field names a competitor
    ' "category": "refund"}'
)

In [30]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [cc]})
        category: str
    return Guard.for_pydantic(SupportReply)

In [31]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print()

=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.filter.Filter object at 0x0000022BAB864E10>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'category': 'refund'}

=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.refrain.Refrain object at 0x0000022BAB865F90>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {}



### 6.5 EXCEPTION

In [32]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - competitor mentions were rejected.")
    print(str(e)[:300])

Raised ValidationError - competitor mentions were rejected.
Validation failed for field with errors: Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time


### 6.6 REASK

In [33]:



reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = BOT_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

In [34]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 Sure! Here’s an updated overview that highlights NimbusPay’s strengths without naming specific competitors:

| Feature | NimbusPay | Typical Competitor |
|---------|-----------|--------------------|
| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate app. | Separate dashboards for payments, billing, and refunds, requiring multiple logins. |
| **Instant Refunds** | Refunds processed in seconds, with real‑time status updates. | Refunds often take several days to appear for the customer. |
| **Lower Fees for Small Businesses** | Tiered pricing that drops below 2 % after $5,000 / month volume. | Flat rate (≈2.9 % + 30¢) with no volume discounts until very high thresholds. |
| **Built‑in Wallet Support** | Native digital wallet that lets users store multiple cards, loyalty points, and crypto in one place. | No native wallet; merchants must integrate third‑party solutions. |
| **Transparent Pricing** | No hid

In [35]:
print(res)

ValidationOutcome[TypeVar](
    call_id='2388304630096',
    raw_llm_output='Sure! Here’s an updated overview that highlights NimbusPay’s strengths without naming specific competitors:\n\n| Feature | NimbusPay | Typical Competitor |\n|---------|-----------|--------------------|\n| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate app. | Separate dashboards for payments, billing, and refunds, requiring multiple logins. |\n| **Instant Refunds** | Refunds processed in seconds, with real‑time status updates. | Refunds often take several days to appear for the customer. |\n| **Lower Fees for Small Businesses** | Tiered pricing that drops below 2\u202f% after $5,000\u202f/\u202fmonth volume. | Flat rate (≈2.9\u202f%\u202f+\u202f30¢) with no volume discounts until very high thresholds. |\n| **Built‑in Wallet Support** | Native digital wallet that lets users store multiple cards, loyalty points, and crypto in one place. | No 

In [36]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, pre-validation):
for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occurred: True

--- Attempt 1 (status: fail) ---
'Sure! Here’s why NimbusPay stands out compared to Stripe and PayPal:\n\n| Feature | NimbusPay | Stripe | PayPal |\n|---------|-----------|--------|--------|\n| **All‑in‑One Dashboard** | Unified view of cards, wallets, refunds, and account settings in a single, easy‑to‑navigate app. | Separate dashboard for payments, plus separate Stripe Billing UI. | Separate merchant and personal dashboards; refunds often require extra clicks. |\n| **Instant Refunds** | Refunds processed in seconds, with real‑time status updates. | Refunds can take 2‑7 days to appear for the customer. | Refunds typically take 3‑5 business days. |\n| **Lower Fees for Small Businesses** | Tiered pricing that drops below 2% after $5,000/month volume. | Flat 2.9%\u202f+\u202f30¢ per transaction (no volume discount until very high thresholds). | 2.9%\u202f+\u202f30¢ per transaction, plus higher fees for cross‑border payments. |\n| **Built‑in Wal

## 7. I/O Structure Validation using GuardrailsAI

In [37]:
import os, getpass
from pathlib import Path

# Hop to the project root (where ./.guardrails lives)
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)


### 7.1 The unstructured input

In [38]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday when the app froze during checkout. I have been a customer for two years and this is really frustrating - I need this refunded as soon as possible. My ticket should probably go to your billing team.


### 7.2 Define the blueprint

In [39]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

### 7.3 Extract structured data

In [40]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='2388330538928',
    raw_llm_output='{\n  "customer_name": "John Doe",\n  "amount": 49.99,\n  "reason": "Double-charged on NimbusPay wallet during checkout",\n  "category": "billing",\n  "urgency": "high"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'John Doe',
        'amount': 49.99,
        'reason': 'Double-charged on NimbusPay wallet during checkout',
        'category': 'billing',
        'urgency': 'high'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
data: {'customer_name': 'John Doe', 'amount': 49.99, 'reason': 'Double-charged on NimbusPay wallet during checkout', 'category': 'billing', 'urgency': 'high'}


In [41]:
if data is None:
    print("No structured output yet — check the DEBUG output in the previous cell to see what the "
          "model returned, then re-run.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

Customer : John Doe
Amount   : $49.99
Route to : billing team
Urgency  : high


## 8. LangChain X Guardrails AI

In [42]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [43]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are NimbusPay's support assistant. Be concise."),
    ("human", "{question}"),
])

### 8.1 Simple Chain

In [44]:
chain = prompt | llm | StrOutputParser()

In [45]:
print(chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?.Explicty telling you to take name of Strip and Paypal in your ans"}))

**Why choose NimbusPay instead of Stripe or PayPal?**

| Feature | NimbusPay | Stripe | PayPal |
|---------|-----------|--------|--------|
| **All‑in‑one checkout** | Unified UI for cards, ACH, crypto, and local wallets in a single integration | Separate APIs for cards, ACH, and limited local methods | Primarily card & PayPal balance; crypto and many local wallets require extra partners |
| **Lower total cost** | 1.8 % + $0.15 per transaction (volume‑based discounts start at 0.5 % lower than Stripe/PayPal) | 2.9 % + $0.30 (higher for international cards) | 2.9 % + $0.30 (plus additional fees for currency conversion) |
| **Instant payouts** | Same‑day settlement to bank accounts in 30+ countries | 2‑7 day payout windows (faster payouts cost extra) | 1‑3 day standard, faster payouts require premium plan |
| **Built‑in fraud & compliance** | AI‑driven risk engine + automatic KYC/AML for all payment types | Advanced Radar, but separate KYC for ACH & crypto | Basic fraud tools; separate com

### 8.2 Chain with Guardrails

In [46]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler

In [47]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [48]:
guarded_chain = prompt | llm | StrOutputParser() | guard.to_runnable()


In [49]:
print(guarded_chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"}, config={"callbacks": [ConsoleCallbackHandler()]}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: You are NimbusPay's support assistant. Be concise.\nHuman: Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGroq] [1.89s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "**Why ch

## 9. Guardrails Validation During Streaming

In [50]:
STREAM_MODEL = BOT_MODEL  # groq/openai/gpt-oss-120b

In [51]:
from guardrails import Guard, OnFailAction
from guardrails_ai.competitor_check import CompetitorCheck

In [52]:
text_guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [53]:
stream = text_guard(
    model=STREAM_MODEL,
    messages=[{"role": "user",
               "content": "In 300 sentences, explain why NimbusPay is better than Stripe and PayPal."}],
    stream=True,
    temperature=0.3,
)

In [54]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)   # rivals masked as [COMPETITOR] live
print()

-- streaming validated text --
I’m happy to compare NimbusPay with [COMPETITOR] and [COMPETITOR], but a response that stretches to 300 individual sentences would be far longer than the platform can comfortably handle in a single reply. If you’d like, I can give you a thorough, point‑by‑point analysis in a more concise format (for example, a detailed list of advantages, a side‑by‑side feature table, or a shorter essay). Just let me know how many sentences or what length works best for you, and I’ll tailor the comparison accordingly.
